In [0]:
%run ./00_logging

DataFrame[]

In [0]:
%run ./01_config_schemas

In [0]:


from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime

# Define widgets
dbutils.widgets.text("batch_date", "2026-10-01")
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"])
dbutils.widgets.dropdown("entity", "all", ["all", "commits", "issues", "pulls"])
dbutils.widgets.text("raw_root", "/Volumes/workspace/landing/raw")

BATCH_DATE = dbutils.widgets.get("batch_date")
LOAD_TYPE  = dbutils.widgets.get("load_type")
ENTITY     = dbutils.widgets.get("entity")
RAW_ROOT   = dbutils.widgets.get("raw_root")

def process_entity(entity):
    start = datetime.now()
    param = f"{LOAD_TYPE}/{BATCH_DATE}/{entity}"
    path = f"{RAW_ROOT}/{LOAD_TYPE}/{BATCH_DATE}/*/{entity}/*.jsonl"
    target = f"workspace.bronze.{entity}"
    stg = f"workspace.bronze._stg_{entity}"
    
    # Verify write_log function is loaded in global scope
    if "write_log" not in globals():
        raise NameError("`write_log` is not defined. Ensure %run ./00_logging executed successfully.")

    try:
        read_schema = StructType(SCHEMAS[entity].fields +
                                 [StructField("_corrupt_record", StringType(), True)])
        
        # Serverless Fix: Write to staging Delta table instead of using .cache()
        (spark.read.schema(read_schema)
              .option("mode", "PERMISSIVE")
              .option("columnNameOfCorruptRecord", "_corrupt_record")
              .json(path)
              .select("*", F.col("_metadata.file_path").alias("source_file"))
              .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(stg))
        
        staged = spark.table(stg)

        # Route bad/corrupt records to quarantine
        bad = staged.filter(F.col("_corrupt_record").isNotNull())
        n_bad = bad.count()
        if n_bad:
            (bad.select(F.lit(entity).alias("entity"),
                        F.lit(BATCH_DATE).alias("batch_date"),
                        F.lit(LOAD_TYPE).alias("load_type"),
                        "source_file",
                        F.col("_corrupt_record").alias("raw_record"),
                        F.current_timestamp().alias("load_timestamp"))
                .write.mode("append").option("mergeSchema", "true")
                .saveAsTable("workspace.bronze.quarantine"))

        # Process valid records
        good = (staged.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
                .withColumn("batch_date", F.lit(BATCH_DATE))
                .withColumn("load_type", F.lit(LOAD_TYPE))
                .withColumn("load_timestamp", F.current_timestamp())
                .dropDuplicates(KEYS[entity]))

        if not spark.catalog.tableExists(target):
            good.limit(0).write.format("delta").saveAsTable(target)

        good.createOrReplaceTempView("src")
        cond = " AND ".join([f"t.{k} = s.{k}" for k in KEYS[entity]])
        
        res = spark.sql(f"""
            MERGE WITH SCHEMA EVOLUTION INTO {target} t
            USING src s ON {cond}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """).collect()[0]

        ins, upd = int(res["num_inserted_rows"]), int(res["num_updated_rows"])
        write_log("raw_to_bronze", entity, param, LOAD_TYPE, start, "SUCCESS",
                  ins, upd, f"quarantined_rows={n_bad}")
        print(f"{entity}: inserted={ins} updated={upd} quarantined={n_bad}")
        
    except Exception as e:
        if "write_log" in globals():
            write_log("raw_to_bronze", entity, param, LOAD_TYPE, start, "FAILURE",
                      0, 0, str(e)[:500])
        print(f"{entity}: FAILED -> {str(e)[:300]}")
    finally:
        spark.sql(f"DROP TABLE IF EXISTS {stg}")

# Execution loop
for e in (["commits", "issues", "pulls"] if ENTITY == "all" else [ENTITY]):
    process_entity(e)

commits: inserted=87 updated=0 quarantined=0
issues: inserted=280 updated=9 quarantined=0
pulls: inserted=232 updated=3 quarantined=0
